In [ ]:
!pip install anthropic -q
import anthropic
import pandas as pd
import json
import time
from google.colab import files, userdata

print('Upload reddit_classified_v1.csv')
uploaded = files.upload()
df = pd.read_csv('Reddit_classified_v1.csv').reset_index(drop=True)
print(f'Reddit rows to reclassify: {len(df)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 6.2 MB/s eta 0:00:00
Upload reddit_classified_v1.csv


Saving Reddit_classified_v1.csv to Reddit_classified_v1.csv
Reddit rows to reclassify: 10620


In [ ]:
# ---------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------
MODEL       = 'claude-haiku-4-5-20251001'
OUTPUT_FILE = 'bumble_reddit_reclassified.csv'

client = anthropic.Anthropic(api_key=userdata.get('bumble_API'))
print(f'Client ready. Model: {MODEL}')

Client ready. Model: claude-haiku-4-5-20251001


In [ ]:
# ---------------------------------------------------------------
# IMPROVED SYSTEM PROMPT
# Key improvements:
# 1. Uses post title + comment together
# 2. Claude assigns sentiment directly
# 3. Clear definitions for easily confused categories
# 4. Strict relevance filter
# ---------------------------------------------------------------

SYSTEM_PROMPT = """You are classifying Reddit comments from dating app subreddits for a consulting analysis of Bumble.

You will be given a POST TITLE and a COMMENT. Use both together to determine the category.
Short comments like "yes exactly" or "this" only make sense with the post title — use it.

RELEVANCE RULE:
If the comment is general life/relationship advice, a personal story unrelated to any app,
a joke, or not meaningfully about a dating app experience — return category_1: "Uncategorised".
If it mentions Bumble, Hinge, Tinder, or dating apps generally, it IS relevant.

CATEGORY DEFINITIONS (choose the best fit):

Dim 1 - Match Quality:
- "No matches" = user is not getting matches/likes/responses at all
- "Bad quality matches" = user IS getting matches but they are poor quality, wrong type, or incompatible
- "Low effort interactions" = matches exist but conversations are one-sided, generic, or copy-pasted openers
- "Ghosting" = conversations started but person disappeared without warning
- "Relationship mismatch" = users want different things (casual vs serious), goals don't align
- "Good quality matches" = positive experience with match quality
- "Relationship success" = met someone, went on dates, started relationship
- "Hookup culture" = frustration that app feels focused on casual sex not relationships
- "Validation seeking" = users using app for ego boost/attention not genuine dating
- "Too many options" = paradox of choice, overwhelmed by number of options

Dim 2 - Emotional Experience:
- "Dating fatigue" = exhausted, burned out, overwhelmed by the process of dating
- "Hopelessness" = given up, feels pointless, deleted app, no hope of finding someone
- "Frustration" = angry or annoyed at specific app behaviour or experience
- "Positive emotional experience" = app is fun, exciting, confidence-boosting
- "Insecurity" = self-doubt, anxiety about appearance or worthiness

Dim 3 - Trust & Safety:
- "Fake profiles & bots" = automated accounts, fake photos, suspicious profiles
- "Scams" = financial scams, romance scams, being deceived for money
- "Harassment & safety" = unwanted messages, creepy behaviour, feeling unsafe
- "Poor support" = app does not respond to reports, bad customer service
- "Good support" = positive customer service experience
- "Account issues" = banned, locked out, account problems
- "Verification" = identity/photo verification features

Dim 4 - Monetisation:
- "Forced subscription" = features locked behind paywall, feels forced to pay
- "Poor value for money" = paid but not worth it, subscription feels useless
- "Good value for money" = premium features are worthwhile
- "Pricing issues" = price increases, refund problems, billing issues
- "Monetisation manipulation" = app artificially limits free users to push upgrades

Dim 5 - Brand & Competition:
- "Worse than competition" = Bumble compared negatively to Hinge/Tinder, switching apps
- "Better than competition" = Bumble preferred over competitors
- "Women-first negative" = women-first mechanic seen as unfair or ineffective
- "Women-first positive" = women-first mechanic appreciated or valued

Dim 6 - Product & UX:
- "UX issues" = confusing interface, poor design, bad user experience
- "UX positive" = app is easy to use, well designed
- "Algorithm issues" = feed showing wrong people, filters not working, profile not shown
- "Bugs" = app crashes, glitches, technical errors

SENTIMENT RULE:
- "negative" = complaint, frustration, bad experience, criticism
- "positive" = praise, satisfaction, success, recommendation
- "neutral" = factual, advice-giving, balanced, no clear emotional direction
Note: sarcasm counts as negative. "Great, another bot" = negative.

Respond ONLY with a JSON object:
{"category_1": "...", "category_2": "..." or null, "sentiment": "positive" or "negative" or "neutral", "confidence": "high" or "medium" or "low"}"""

def build_message(post_title, comment_text):
    title = str(post_title) if pd.notna(post_title) else 'No title'
    text  = str(comment_text)[:500]
    return f"POST TITLE: {title}\n\nCOMMENT: {text}"

print('Prompt ready.')

Prompt ready.


In [ ]:
# ---------------------------------------------------------------
# TEST ON 10 ROWS BEFORE FULL BATCH
# ---------------------------------------------------------------
print('Testing on 10 rows...\n')
test_sample = df.sample(10, random_state=42)

for i, (_, row) in enumerate(test_sample.iterrows()):
    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=150,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": build_message(row['post_title'], row['text'])}]
        )
        raw = response.content[0].text.strip()
        raw = raw.replace('```json', '').replace('```', '').strip()
        result = json.loads(raw)
        print(f"Row {i+1}:")
        print(f"  Post title: {str(row['post_title'])[:60]}")
        print(f"  Comment:    {str(row['text'])[:80]}")
        print(f"  Old category: {row['category_1']} | New: {result}")
        print()
    except Exception as e:
        print(f"Row {i+1}: Error — {e}")

print('Test complete. Check results above before running full batch.')

Testing on 10 rows...

Row 1:
  Post title: If you are a man how many matches do you get on average?
  Comment:    I live in Berlin and the dating scene here is huge and a lot of the time really 
  Old category: Dating fatigue | New: {'category_1': 'No matches', 'category_2': 'Dating fatigue', 'sentiment': 'negative', 'confidence': 'medium'}

Row 2: Error — Extra data: line 9 column 1 (char 115)
Row 3: Error — Extra data: line 9 column 1 (char 113)
Row 4: Error — Extra data: line 9 column 1 (char 112)
Row 5: Error — Extra data: line 9 column 1 (char 122)
Row 6:
  Post title: Are you all really meeting the love of your life on Bumble o
  Comment:    My sister also met her husband on Tinder and my best girlfriends met their husba
  Old category: Relationship success | New: {'category_1': 'Relationship success', 'category_2': None, 'sentiment': 'positive', 'confidence': 'high'}

Row 7:
  Post title: Honest opinion needed
  Comment:    i’d swipe right, but it has more to do w my desperatio

In [ ]:
# ---------------------------------------------------------------
# SUBMIT FULL BATCH
# ---------------------------------------------------------------
print(f'Building {len(df)} batch requests...')

requests = []
for idx, row in df.iterrows():
    requests.append({
        "custom_id": str(idx),
        "params": {
            "model":    MODEL,
            "max_tokens": 150,
            "system":   SYSTEM_PROMPT,
            "messages": [{"role": "user", "content": build_message(row['post_title'], row['text'])}]
        }
    })

print(f'Submitting batch...')
batch    = client.messages.batches.create(requests=requests)
BATCH_ID = batch.id
print(f'Batch submitted!')
print(f'BATCH_ID: {BATCH_ID}')
print(f'Status: {batch.processing_status}')
print(f'IMPORTANT: Save this Batch ID in your Colab secrets as BATCH_ID_REDDIT')

Building 10620 batch requests...
Submitting batch...
Batch submitted!
BATCH_ID: msgbatch_019vNnscqk7hnWfTjhW7uEkA
Status: in_progress
IMPORTANT: Save this Batch ID in your Colab secrets as BATCH_ID_REDDIT


In [ ]:
# ---------------------------------------------------------------
# CHECK STATUS — re-run this cell until Status: ended
# If session reset: BATCH_ID = userdata.get('BATCH_ID_REDDIT')
# ---------------------------------------------------------------
# If session reset, restore from secret:
BATCH_IDS = [userdata.get('BATCH_ID_REDDIT')]

all_done        = True
total_succeeded = 0
total_pending   = 0
total_errored   = 0

for bid in BATCH_IDS:
    status = client.messages.batches.retrieve(bid)
    print(f'{bid}: {status.processing_status} — succeeded: {status.request_counts.succeeded}, pending: {status.request_counts.processing}')
    total_succeeded += status.request_counts.succeeded
    total_pending   += status.request_counts.processing
    total_errored   += status.request_counts.errored
    if status.processing_status != 'ended':
        all_done = False

print(f'\nTotal succeeded: {total_succeeded}')
print(f'Total pending:   {total_pending}')
print(f'All complete:    {all_done}')

msgbatch_016rCAqmLZHsz3vGLHGjodUT: ended — succeeded: 10620, pending: 0

Total succeeded: 10620
Total pending:   0
All complete:    True


In [ ]:
# If session reset, restore from secret:
BATCH_IDS = [userdata.get('BATCH_ID_REDDIT')]

print('Retrieving results...')
results = {}
for bid in BATCH_IDS:
    for result in client.messages.batches.results(bid):
        idx = int(result.custom_id)
        if result.result.type == 'succeeded':
            try:
                raw = result.result.message.content[0].text.strip()
                raw = raw.replace('```json', '').replace('```', '').strip()
                parsed = json.loads(raw)
                results[idx] = parsed
            except:
                results[idx] = {"category_1": "Uncategorised", "category_2": None,
                                "sentiment": "neutral", "confidence": "low"}
        else:
            results[idx] = {"category_1": "Uncategorised", "category_2": None,
                            "sentiment": "neutral", "confidence": "low"}

print(f'Retrieved {len(results)} results')

Retrieving results...
Retrieved 10620 results


In [ ]:
# ---------------------------------------------------------------
# BUILD RECLASSIFIED REDDIT DATAFRAME
# ---------------------------------------------------------------
DIMENSION_MAP = {
    'Good quality matches':          'Dim 1 - Match Quality',
    'Bad quality matches':           'Dim 1 - Match Quality',
    'Ghosting':                      'Dim 1 - Match Quality',
    'Low effort interactions':       'Dim 1 - Match Quality',
    'Hookup culture':                'Dim 1 - Match Quality',
    'Relationship success':          'Dim 1 - Match Quality',
    'Relationship mismatch':         'Dim 1 - Match Quality',
    'Validation seeking':            'Dim 1 - Match Quality',
    'No matches':                    'Dim 1 - Match Quality',
    'Too many options':              'Dim 1 - Match Quality',
    'Dating fatigue':                'Dim 2 - Emotional Experience',
    'Hopelessness':                  'Dim 2 - Emotional Experience',
    'Frustration':                   'Dim 2 - Emotional Experience',
    'Positive emotional experience': 'Dim 2 - Emotional Experience',
    'Insecurity':                    'Dim 2 - Emotional Experience',
    'Fake profiles & bots':          'Dim 3 - Trust & Safety',
    'Scams':                         'Dim 3 - Trust & Safety',
    'Harassment & safety':           'Dim 3 - Trust & Safety',
    'Verification':                  'Dim 3 - Trust & Safety',
    'Good support':                  'Dim 3 - Trust & Safety',
    'Poor support':                  'Dim 3 - Trust & Safety',
    'Account issues':                'Dim 3 - Trust & Safety',
    'Forced subscription':           'Dim 4 - Monetisation',
    'Poor value for money':          'Dim 4 - Monetisation',
    'Good value for money':          'Dim 4 - Monetisation',
    'Pricing issues':                'Dim 4 - Monetisation',
    'Monetisation manipulation':     'Dim 4 - Monetisation',
    'Better than competition':       'Dim 5 - Brand & Competition',
    'Worse than competition':        'Dim 5 - Brand & Competition',
    'Women-first positive':          'Dim 5 - Brand & Competition',
    'Women-first negative':          'Dim 5 - Brand & Competition',
    'UX issues':                     'Dim 6 - Product & UX',
    'UX positive':                   'Dim 6 - Product & UX',
    'Algorithm issues':              'Dim 6 - Product & UX',
    'Bugs':                          'Dim 6 - Product & UX',
    'Uncategorised':                 'Uncategorised',
}

df['category_1']               = df.index.map(lambda i: results.get(i, {}).get('category_1', 'Uncategorised'))
df['category_2']               = df.index.map(lambda i: results.get(i, {}).get('category_2'))
df['sentiment_label']          = df.index.map(lambda i: results.get(i, {}).get('sentiment', 'neutral'))
df['classification_confidence']= df.index.map(lambda i: results.get(i, {}).get('confidence', 'low'))
df['dimension_1']              = df['category_1'].map(DIMENSION_MAP)
df['dimension_2']              = df['category_2'].map(DIMENSION_MAP)

print(f'Reclassified {len(df)} Reddit rows')
print(f'\nCategory distribution (top 15):')
print(df['category_1'].value_counts().head(15).to_string())
print(f'\nSentiment distribution:')
print(df['sentiment_label'].value_counts().to_string())
print(f'\nUncategorised: {(df["category_1"] == "Uncategorised").sum()} rows ({(df["category_1"] == "Uncategorised").mean()*100:.1f}%)')

Reclassified 10620 Reddit rows

Category distribution (top 15):
category_1
Uncategorised              6025
Low effort interactions     818
No matches                  790
Bad quality matches         502
Ghosting                    320
Relationship mismatch       230
Algorithm issues            223
Relationship success        221
Good quality matches        209
Fake profiles & bots        176
Account issues              132
Better than competition      98
Dating fatigue               86
Too many options             76
Monetisation                 75

Sentiment distribution:
sentiment_label
neutral     7620
negative    2471
positive     529

Uncategorised: 6025 rows (56.7%)


In [ ]:
df.to_csv('bumble_reddit_reclassified.csv', index=False)
print(f'Saved bumble_reddit_reclassified.csv ({len(df)} rows)')

Saved bumble_reddit_reclassified.csv (10620 rows)


In [ ]:
files.download('bumble_reddit_reclassified.csv')
print('Download triggered.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered.
